This is a Notebook file where we evaluate the models using different graphs, metrics and visualisations. 

In [87]:
import pandas as pd
import numpy as np

import pickle

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    roc_curve,
    auc,
)

path_png = "D:/Users/nickt/ECO482/Research Project Files/Visualizations/pngs"

Load in the model results from models.py

In [88]:
model_path = "D:/Users/nickt/ECO482/Research Project Files/Models/trained_models.pkl"

with open(model_path, "rb") as f:
    saved_data = pickle.load(f)

best_models = saved_data["best_models"]
summary_df = saved_data["summary_df"]
evaluation_results = saved_data["evaluation_results"]
X = saved_data["X"]
y = saved_data["y"]
feature_names = saved_data["feature_names"]

Summary Statistics for Each Model

In [89]:
results_df = pd.DataFrame(evaluation_results)
summary_table = results_df.groupby('model')[['accuracy', 'recall', 'precision', 'roc_auc', 'avg_precision']].mean()
summary_std = results_df.groupby('model')[['accuracy', 'recall', 'precision', 'roc_auc', 'avg_precision']].std()

summary_display = results_df[['model', 'fold', 'accuracy', 'recall', 'precision', 'roc_auc', 'avg_precision']].copy()
summary_display = summary_display.round(4)

print("MODEL EVALUATION SUMMARY")
print(summary_display.to_string(index=False))
print("\nAVERAGE PERFORMANCE BY MODEL:")
print(summary_table.round(4))
print("\nSTANDARD DEVIATION BY MODEL:")
print(summary_std.round(4))

MODEL EVALUATION SUMMARY
        model  fold  accuracy  recall  precision  roc_auc  avg_precision
      XGBoost     1    0.8683  0.7143     0.3030   0.8889         0.3610
      XGBoost     2    0.9415  0.7667     0.8214   0.9646         0.8587
Random Forest     1    0.8732  0.7143     0.3125   0.8773         0.2970
Random Forest     2    0.9268  0.6667     0.8000   0.9484         0.7820
  Random Walk     1    0.8927  0.2143     0.2143      NaN            NaN
  Random Walk     2    0.9220  0.6667     0.7692      NaN            NaN

AVERAGE PERFORMANCE BY MODEL:
               accuracy  recall  precision  roc_auc  avg_precision
model                                                             
Random Forest    0.9000  0.6905     0.5562   0.9129         0.5395
Random Walk      0.9073  0.4405     0.4918      NaN            NaN
XGBoost          0.9049  0.7405     0.5622   0.9268         0.6098

STANDARD DEVIATION BY MODEL:
               accuracy  recall  precision  roc_auc  avg_precision
m

Below we create visualizations based on each models performance. We first create and compare ROC Curves for each Model

In [90]:
roc_fig = go.Figure()

for model_name in best_models.keys():
    model_results = [r for r in evaluation_results if r['model'] == model_name]
    y_true = np.concatenate([r['y_true'] for r in model_results])
    y_score = np.concatenate([r['y_proba'] for r in model_results])

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    roc_fig.add_trace(go.Scatter(
        x=fpr, 
        y=tpr, 
        mode='lines', 
        name=f'{model_name} (AUC={roc_auc:.3f})')
    )

roc_fig.add_trace(go.Scatter(
    x=[0, 1], 
    y=[0, 1], 
    mode='lines', 
    name='Random Chance', 
    line=dict(dash='dash', color='gray'))
    )

roc_fig.update_layout(
    title={'text':'ROC Curves', 'x':0.5, 'xanchor':'center'}, 
    xaxis_title='False Positive Rate', 
    yaxis_title='True Positive Rate', 
    width=900,
    height=500, 
    template='simple_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="right",
        x=0.9
    ), 
    font=dict(size=14),
)

roc_fig.show()
roc_fig.write_image(f"{path_png}/roc_curves.png", scale=2)

Precision Recall Curves

In [91]:
pr_fig = go.Figure()

for model_name in best_models.keys():
    model_results = [r for r in evaluation_results if r['model'] == model_name]
    y_true = np.concatenate([r['y_true'] for r in model_results])
    y_score = np.concatenate([r['y_proba'] for r in model_results])

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    avg_prec = average_precision_score(y_true, y_score)
    pr_fig.add_trace(go.Scatter(
        x=recall, 
        y=precision, 
        mode='lines', 
        name=f'{model_name} (AP={avg_prec:.3f})')
    )

pr_fig.update_layout(
    title={'text':'Precision-Recall Curves', 'x':0.5, 'xanchor':'center'}, 
    xaxis_title='Recall', 
    yaxis_title='Precision', 
    width=900,
    height=500, 
    template='simple_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="right",
        x=0.8
    ), 
    font=dict(size=14),
)

pr_fig.show()
pr_fig.write_image(f"{path_png}/pr_curves.png", scale=2)

In [92]:
cm_list = []

model_names = list(best_models.keys()) + ['Random Walk']

for model_name in model_names:
    model_results = [r for r in evaluation_results if r['model'] == model_name]
    y_true = np.concatenate([r['y_true'] for r in model_results])
    y_pred = np.concatenate([r['y_pred'] for r in model_results])
    cm = confusion_matrix(y_true, y_pred)
    cm_list.append((model_name, cm))

shared_max = max(cm.max() for _, cm in cm_list)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[f'{model_name}' for model_name, _ in cm_list],
    horizontal_spacing=0.10,
)
for ann in fig.layout.annotations:
    ann.font = dict(size = 18)
    ann.yshift = 15

n_cols = len(cm_list)

for idx, (model_name, cm) in enumerate(cm_list, start=1):
    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=['No Inversion', 'Inversion'],
            y=['No Inversion', 'Inversion'],
            text=cm,
            texttemplate='%{text}',
            colorscale='Cividis',
            zmin=0,
            zmax=shared_max,
            showscale=(idx == n_cols),
            colorbar=dict(
                x=1.02,    
                y=0.5,
            ),
        ),
        row=1,
        col=idx,
    )

fig.update_layout(
    width=1500,
    height=500,
    template='plotly_white',
    margin=dict(l=90, r=100, t=110, b=90),
    annotations=list(fig.layout.annotations) + [
        dict(
            text='Predicted label',
            x=0.5, y=-0.16,
            xref='paper', yref='paper',
            showarrow=False,
            font=dict(size=15),
        ),
        dict(
            text='True label',
            x=-0.07, y=0.5,
            xref='paper', yref='paper',
            showarrow=False,
            textangle=-90,
            font=dict(size=15),
        ),
    ],
)

fig.show()
fig.write_image(f"{path_png}/confusion_matrices.png", scale=2)

Feature Importance

In [93]:
model_specs = [("XGBoost", "#4C78A8"), ("Random Forest", "#F58518")]

for model_name, color in model_specs:
    importances = best_models[model_name].feature_importances_
    feature_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances,
    }).sort_values(by='Importance', ascending=False)
    top_features = feature_df.head(15)[::-1]

    fig = px.bar(
        top_features,
        x='Importance',
        y='Feature',
        orientation='h',
        title=f'{model_name} Feature Importance',
        labels={'Importance': 'Importance', 'Feature': 'Feature'}
    )
    fig.update_traces(
        marker_color=color,
        marker_line_color='black',
        marker_line_width=1,
        marker_pattern_shape='/'
    )

    fig.update_layout(
        title={'x': 0.5, 'xanchor': 'center'},
        yaxis={'categoryorder': 'total ascending'},
        width=900,
        height=700,
        template='plotly_white',
        font=dict(size=14),
    )
    
    fig.show()
    fig.write_image(f"{path_png}/{model_name.lower().replace(' ', '_')}_feature_importance.png", scale=2)